# RAG 기반 챗봇 구현하기

## 필요 라이브러리 설치
- 가상 환경에 설치 권장
  - 가상 환경 활성화
  - 각 라이브러리 설치
  - 해당 가상 환경 커널 설정 후 실행

In [2]:
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install langchain==0.1.12

  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached packaging-23.2-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_core-0.1.53-py3-none-any.whl (303 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
Using cached packaging-23.2-py3-none-any.whl (53 kB)

  Attempting uninstall: packaging

    Found existing installation: packaging 26.3

    Uninstalling packaging-26.3:

      Successfully uninstalled packaging-26.3

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.14.17

   ------------- -------------------------- 1/3 [langsmith]
    Uninstalling langsmith-0.14.17:
   ------------- -------------------------- 1/3 [langsmith]
   ------------- -------------------------- 1/3 [langsmith]
      Successfully uninstalled langsmith-0.14.17
   ------------- -------------------------- 1/3 [langsmith]
   ------------- -------------------------- 1/3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
build 1.6.1 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.1.53 which is incompatible.
langchain-openai 1.7.3 requires langchain-core<2.0.0,>=1.7.0, but you have langchain-core 0.1.53 which is incompatible.
langgraph 1.2.19 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.1.53 which is incompatible.
langgraph-checkpoint 4.3.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.1.53 which is incompatible.
langgraph-sdk 0.4.11 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.1.53 which is incompatible.
streamlit 1.41.1 requires protobuf<6,>=

In [5]:
%pip install langchain-community

Note: you may need to restart the kernel to use updated packages.


## PDF 파일 텍스트 변환 및 청크 쪼개기

### PDF 파일을 읽어서 텍스트 데이터 추출

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader('./data/OneNYC_2050_Strategic_Plan.pdf')
data_nyc = loader.load()
data_nyc[0]

Document(metadata={'source': './data/OneNYC_2050_Strategic_Plan.pdf', 'file_path': './data/OneNYC_2050_Strategic_Plan.pdf', 'page': 0, 'total_pages': 332, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe InDesign 14.0 (Windows)', 'producer': 'Adobe PDF Library 15.0', 'creationDate': "D:20190430134830-04'00'", 'modDate': "D:20200103155512-05'00'", 'trapped': ''}, page_content='OneNYC \n2050\nBUILDING A STRONG \nAND FAIR CITY \nVOLUME 1 OF 9\t\t\nAPRIL 2019\nTHE CITY OF NEW YORK\nMAYOR BILL DE BLASIO\nDEAN FULEIHAN \nFIRST DEPUTY MAYOR \nDOMINIC WILLIAMS \nCHIEF POLICY ADVISOR\nDANIEL A. ZARRILLI  \nOneNYC DIRECTOR\n')

### Splitter 설정 및 청크 쪼개기
- 오버랩 : 청크 경계에서 문맥이 끊기는 것을 방지

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

In [6]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

Split 1:------------------------------------

page_content='OneNYC 
2050
BUILDING A STRONG 
AND FAIR CITY 
VOLUME 1 OF 9		
APRIL 2019
THE CITY OF NEW YORK
MAYOR BILL DE BLASIO
DEAN FULEIHAN 
FIRST DEPUTY MAYOR 
DOMINIC WILLIAMS 
CHIEF POLICY ADVISOR
DANIEL A. ZARRILLI  
OneNYC DIRECTOR' metadata={'source': './data/OneNYC_2050_Strategic_Plan.pdf', 'file_path': './data/OneNYC_2050_Strategic_Plan.pdf', 'page': 0, 'total_pages': 332, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe InDesign 14.0 (Windows)', 'producer': 'Adobe PDF Library 15.0', 'creationDate': "D:20190430134830-04'00'", 'modDate': "D:20200103155512-05'00'", 'trapped': ''}
Split 2:------------------------------------

page_content='2   |   OneNYC 2050 
NYC.GOV/OneNYC
ONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S 
FUTURE AGAINST THE CHALLENGES OF TODAY AND 
TOMORROW. WITH BOLD ACTIONS TO CONFRONT OUR 
CLIMATE CRISIS, ACHIEVE EQUITY, AND STRENGTHEN 
OUR DEMOCRACY, WE ARE BUILDI

In [7]:
print(type(all_splits[0]))

<class 'langchain_core.documents.base.Document'>


### 다른 PDF 파일도 동일 작업 진행

In [8]:
loader_seoul = PyMuPDFLoader('./data/2040_seoul_plan.pdf')
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)
for i, split in enumerate(seoul_splits):
    print(f"Split {i+1}:------------------------------------")
    print(split)

Split 1:------------------------------------
page_content='「2040 서울도시기본계획」을 발간하며
지난 3년간 코로나19 팬데믹으로 전 세계가 심각한 타격을 받아왔지만, 대한민국의 수도 서
울은 혁신적인 디지털 기술과 뛰어난 시민 의식, 풍부한 자연환경을 토대로 도시의 가능성과 
잠재력을 확인할 수 있었습니다. 「2040 서울도시기본계획」은 기후위기, 디지털 전환, 생활양
식의 변화 등 글로벌 대도시가 당면한 과제에 대한 해법을 제시하고 있습니다.
첫째, 보행일상권으로의 공간구조 개편입니다. 서울을 하나의 기준으로 관리하던 지금까지의 
방식에서 벗어나, 미래의 서울은 다양한 지역특성을 반영한 차별화된 계획체계를 통해 동네
단위의 자족적 생활권으로 재구성하게 됩니다. 이는 기후위기, 팬데믹 등 각종 재난상황에서
도 도시활동이 가능한 공간단위로, 서울 어디에서나 시민 삶의 질이 보장되는 새로운 차원의 
균형발전정책 기반이 될 것입니다.
둘째, 과감하고 유연한 도시계획 기조로의 전환입니다. 앞으로의 도시계획은 미래 여건변화
에 유연하고 신속하게 대응할 수 있는 체계를 통해 현장에서 강력하게 작동하는 수단이 될 것
입니다. 미래지향적인 계획철학을 토대로 융복합적인 토지이용제도를 실현하고 수변녹지와 
연계된 생활공간을 조성하는 등 도시계획체계를 과감하고 유연하게 전환했습니다.' metadata={'source': './data/2040_seoul_plan.pdf', 'file_path': './data/2040_seoul_plan.pdf', 'page': 1, 'total_pages': 205, 'format': 'PDF 1.4', 'title': '', 'author': 'SI', 'subject': '', 'keywords': '', 'creator': 'Hwp 2020 11.0.0.5178', 'producer': 'Hancom PDF 1.3.0.542', 'creationDate': "D:20241212181611+

In [9]:
print(seoul_splits[50].page_content)
print('----------------------')
print(seoul_splits[51].page_content)

34
제2장 미래상과 목표
3. 서울의 미래 여건 변화와 과제
1) 가속화되는 저출생·고령화
2040년 서울의 고령인구는 약 32%, 늘어나는 복지·의료 
수요 대응 필요
Ÿ 장기적인 저출생·고령화로 인해 인구변화 속도는 둔화되는 
반면, 기술발전에 따른 평균수명 증가로 고령인구의 비율은 
지속적으로 상승 중이다. 
Ÿ 서울의 고령화 속도는 빠른 편으로 노년인구 비율은 2018년 
기준 14.4%로 고령사회에 진입하였으며, 2026년에는 초
고령화사회에 진입할 것으로 예측된다.
2040년 서울도시기본계획 계획인구는 통계청 전망을 반영한 854만 명으로 설정
Ÿ 2040 서울도시기본계획의 계획인구는 장기적으로 발생하고 있는 인구 감소와 고령화, 저성장 등 도시
환경 변화를 반영하는 지표로 도시의 환경용량, 인프라 수요 등을 결정
Ÿ 2040년 계획인구는 다음의 근거를 고려하여 통계청의 추계인구(중위)인 854만 명으로 설정
- 첫째, 광역 교통망의 발달로 수도권은 이미 하나의 생활권이 되었으며, 향후 수도권 광역급행철도 등 
신규 교통수단의 발달로 광역 생활권의 범위 확대는 더욱 가속화될 것으로 전망
- 둘째, 서울의 인구는 감소할 것으로 전망되지만 서울로 오가며 생활하는 인구는 오히려 늘어날 것으로 
전망되어 거주 인구로만은 판단하기 어려운 복잡한 인구이동 특성을 보유
Ÿ 서울 대도시권으로의 공간적 확장, 인접지역 간 발생하는 생활인구 등 복잡한 현황을 고려하여 인구감소 
추세에 대응하고 시민 삶의 질 향상을 위한 계획인구를 설정
2) 일상의 반강제적 디지털 전환에 대응
디지털 전환에 따른 시민 일상생활 및 도시공간 변화 감지
Ÿ 2019년 갑작스럽게 발생한 코로나19는 도시의 산업구조뿐만 아니라 일상에서 반강제적 디지털 전환
을 가속화하였다.
- 원격의료, 재택근무, 온라인 교육 등 다양한 온라인 기반 플랫폼의 등장
- 중급역량4)이 2021년 기준, 2019년 대비 10.1% 증가한 74.1%로 높은 증가 추세
----------------------
- 

### 후처리 과정
- 현재 청크($i$)의 끝에 다음 청크($i+1$)의 시작 100자를 덧붙임
- LSM이 다음 청크로 넘어가기 전의 문맥을 잃지 않고 답변을 생성

In [10]:
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += "\n"+ seoul_splits[i + 1].page_content[:100]

print(seoul_splits[50].page_content)
print('----------------------')
print(seoul_splits[51].page_content)

34
제2장 미래상과 목표
3. 서울의 미래 여건 변화와 과제
1) 가속화되는 저출생·고령화
2040년 서울의 고령인구는 약 32%, 늘어나는 복지·의료 
수요 대응 필요
Ÿ 장기적인 저출생·고령화로 인해 인구변화 속도는 둔화되는 
반면, 기술발전에 따른 평균수명 증가로 고령인구의 비율은 
지속적으로 상승 중이다. 
Ÿ 서울의 고령화 속도는 빠른 편으로 노년인구 비율은 2018년 
기준 14.4%로 고령사회에 진입하였으며, 2026년에는 초
고령화사회에 진입할 것으로 예측된다.
2040년 서울도시기본계획 계획인구는 통계청 전망을 반영한 854만 명으로 설정
Ÿ 2040 서울도시기본계획의 계획인구는 장기적으로 발생하고 있는 인구 감소와 고령화, 저성장 등 도시
환경 변화를 반영하는 지표로 도시의 환경용량, 인프라 수요 등을 결정
Ÿ 2040년 계획인구는 다음의 근거를 고려하여 통계청의 추계인구(중위)인 854만 명으로 설정
- 첫째, 광역 교통망의 발달로 수도권은 이미 하나의 생활권이 되었으며, 향후 수도권 광역급행철도 등 
신규 교통수단의 발달로 광역 생활권의 범위 확대는 더욱 가속화될 것으로 전망
- 둘째, 서울의 인구는 감소할 것으로 전망되지만 서울로 오가며 생활하는 인구는 오히려 늘어날 것으로 
전망되어 거주 인구로만은 판단하기 어려운 복잡한 인구이동 특성을 보유
Ÿ 서울 대도시권으로의 공간적 확장, 인접지역 간 발생하는 생활인구 등 복잡한 현황을 고려하여 인구감소 
추세에 대응하고 시민 삶의 질 향상을 위한 계획인구를 설정
2) 일상의 반강제적 디지털 전환에 대응
디지털 전환에 따른 시민 일상생활 및 도시공간 변화 감지
Ÿ 2019년 갑작스럽게 발생한 코로나19는 도시의 산업구조뿐만 아니라 일상에서 반강제적 디지털 전환
을 가속화하였다.
- 원격의료, 재택근무, 온라인 교육 등 다양한 온라인 기반 플랫폼의 등장
- 중급역량4)이 2021년 기준, 2019년 대비 10.1% 증가한 74.1%로 높은 증가 추세
- 원격의료, 재택근무, 온라인 교육 등 다양

In [11]:
print(len(all_splits))
all_splits.extend(seoul_splits)
print(len(all_splits))

1016
1305


## 오픈 AI 임베딩 모델 사용하기

### 벡터DB 다운로드

In [14]:
%pip install langchain_chroma

  Using cached langchain_core-1.7.6-py3-none-any.whl.metadata (4.8 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached python_dateutil-4.0.0-py3-none-any.whl.metadata (7.3 kB)
Using cached langchain_core-1.7.6-py3-none-any.whl (580 kB)
Using cached langsmith-0.14.17-py3-none-any.whl (940 kB)
Using cached packaging-26.3-py3-none-any.whl (129 kB)
Using cached python_dateutil-4.0.0-py3-none-any.whl (92 kB)

  Attempting uninstall: python-dateutil

    Found existing installation: python-dateutil None

   ---------------------------------------- 0/4 [python-dateutil]

Note: you may need to restart the kernel to use updated packages.


error: uninstall-no-record-file

× Cannot uninstall python-dateutil None
╰─> The package's contents are unknown: no RECORD file was found for python-dateutil.

hint: You might be able to recover from this via: pip install --ignore-installed --no-deps python-dateutil==2.9.0.post0


In [15]:
%pip install langchain_openai

  Using cached langchain_core-1.7.6-py3-none-any.whl.metadata (4.8 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
Using cached langchain_core-1.7.6-py3-none-any.whl (580 kB)
Using cached langsmith-0.14.17-py3-none-any.whl (940 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.1.147

    Uninstalling langsmith-0.1.147:

      Successfully uninstalled langsmith-0.1.147

   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------------------- 0/2 [langsmith]
   ---------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.12 requires langchain-core<0.2.0,>=0.1.31, but you have langchain-core 1.7.6 which is incompatible.
langchain 0.1.12 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.14.17 which is incompatible.
langchain-community 0.0.38 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.7.6 which is incompatible.
langchain-community 0.0.38 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.14.17 which is incompatible.
langchain-text-splitters 0.0.2 requires langchain-core<0.3,>=0.1.28, but you have langchain-core 1.7.6 which is incompatible.


### 텍스트를 벡터로 변환

In [1]:
from langchain_openai import OpenAIEmbeddings

In [2]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
print(v)
print(len(v))

[-0.00536346435546875, -0.049530029296875, -0.01250457763671875, 0.00847625732421875, -0.040069580078125, -0.0104217529296875, 0.0063018798828125, 0.025421142578125, -0.049774169921875, -0.0123291015625, -0.0019893646240234375, -0.017822265625, 0.026275634765625, -0.0010824203491210938, 0.00240325927734375, -0.0182342529296875, -0.0389404296875, -0.043182373046875, 0.07275390625, -0.012054443359375, 0.0494384765625, -0.0147857666015625, -0.0191192626953125, 0.0059967041015625, -0.0328369140625, 0.0169830322265625, 0.0250396728515625, 0.02020263671875, -0.004241943359375, 0.037750244140625, 0.035736083984375, 0.03057861328125, 0.0175628662109375, -0.00931549072265625, -0.0178985595703125, 0.01371002197265625, 0.06256103515625, -0.03033447265625, 0.04901123046875, -0.00263214111328125, -0.0274200439453125, 0.0129241943359375, -0.032989501953125, -0.00745391845703125, 0.0195770263671875, -0.0031528472900390625, 0.01071929931640625, -0.03094482421875, -0.0082550048828125, -0.00547790527343

### Chroma DB 생성 및 불러오기

In [12]:
from langchain_chroma import Chroma
import os

persist_directory = '../chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

Creating new Chroma store


### 유사 청크 가져와서 질문하기

In [13]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책에 대해 궁금해")

for d in docs:
    print(d)
    print('------')

page_content='제3절 경제·산업 부문
77
2-4 공정한 경제 환경 조성을 위한 상생협력 생태계 조성
2-4-1 경제주체 간 상생 생태계 강화 및 공동 협업 확대 
Ÿ 서울의 산업구조가 과거 제조업 중심에서 서비스업 중심으로 변화되면서 제조업 기반의 노동집약적 
산업들이 감소하고 있다. 산업구조의 지속가능성과 경쟁력을 갖추기 위해, 분야별 최소한의 지표 설정 
및 모니터링을 통해 산업의 다양성을 유지한다.
Ÿ 경제주체 간 상생 생태계 강화와 공동사업, 공동브랜드, 공동판매 등 협업을 강조하고 이를 정책적
으로 지원한다.
Ÿ 전통산업과 다양한 서비스업 간 연계를 활성화하고 이를 촉진하기 위한 권역별 앵커시설을 조성하여 
고부가가치를 창출할 수 있는 기반을 마련한다.
Ÿ 서비스업이 전통산업을 지원하도록 인센티브 마련과 규제완화 등 관련 제도를 정비한다.
2-4-2 노동 취약계층의 사각지대 해소 및 선제적 지원 제공
Ÿ 사회변화 및 기술발달에 따라 새로운 노동형태가 출현하는 변화의 흐름에 유연하게 대응할 수 있도록 
플랫폼 노동자, 프리랜서 등과 같은 특수노동자들의 법적 사각지대를 해소하기 위한 보호 정책을 
마련한다.
- 사회보험을 지원하고 병가와 휴가비 등 서울시 유급병가(보건의료) 및 유급휴가(문화관광) 지원을 확대
Ÿ 높은 수준의 고용의 질과 근로환경 속에서 정당한 대가와 노동인권 및 권익을 존중받을 수 있는 
기업문화를 조성하기 위한 지원방안을 마련한다.
2-4-3 소상공인·자영업자의 사업 비용부담 완화 및 사회안전망 강화 
Ÿ 소상공인·자영업자가 사업 도중 불가피한 상황이 발생했을 시, 이를 지원하고 보호하는 공적 안전망 
제도 확충을 통해 사업 안정을 도모한다.
78
제3장 부문별 전략계획
제4절 
기후·환경 부문
1. 개요
Ÿ 기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측
면과 연관되' metadata={'trapped': '', 'keywords': '', 'format': 'PDF 1

## 청크 기반으로 답변 생성하기

### 의존성 문제 해결을 위해 라이브러리 추가 설치

In [14]:
%pip install langchain-classic

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 1.4 MB/s eta 0:00:01
   ------------------------------ --------- 0.8/1.0 MB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 1.2 MB/s  0:00:00

  Attempting uninstall: langchain-text-splitters

    Found existing installation: langchain-text-splitters 0.0.2

    Uninstalling langchain-text-splitters-0.0.2:

      Successfully uninstalled langchain-text-splitters-0.0.2

   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.12 requires langchain-core<0.2.0,>=0.1.31, but you have langchain-core 1.7.6 which is incompatible.
langchain 0.1.12 requires langchain-text-splitters<0.1,>=0.0.1, but you have langchain-text-splitters 1.1.3 which is incompatible.
langchain 0.1.12 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.14.17 which is incompatible.


### 청크를 바탕으로 답변하는 시스템 생성하기

In [15]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")


question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

## 메세지 저장하고 답변 생성하기

In [16]:
from langchain_classic.memory import ChatMessageHistory

# 채팅 메시지 저장할 메모리 객체 생성
chat_history = ChatMessageHistory() 
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.") 

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

C:\Users\Administer\AppData\Local\Temp\ipykernel_20524\2532468324.py:4: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history = ChatMessageHistory()


서울시는 온실가스 저감을 위해 여러 가지 정책과 전략을 추진하고 있습니다. 주요 내용은 다음과 같습니다:

1. **탄소중립 목표 설정**: 2050년까지 탄소중립을 달성하기 위해 도시 인프라의 전환이 필요하며, 이를 위한 다양한 친환경 기술 개발 및 적용이 강조되고 있습니다.

2. **건물 부문 탄소배출 감축**: 친환경 건축 기술을 개발하고 이를 적극적으로 적용하여 건물에서 발생하는 탄소배출을 줄이는 노력이 이루어지고 있습니다.

3. **교통 및 에너지 혁신**: 미래 모빌리티 기술을 활용하고 친환경 차량 및 관련 인프라를 확충하여 교통 부문에서도 온실가스 배출을 감축할 계획입니다. 또한, 청정에너지를 기반으로 에너지 전환을 추진하고 있습니다.

4. **대기 환경 관리**: 대기 환경을 고려한 공간계획과 배출원 관리체계를 강화하여 전체적인 대기 질 개선에 기여하고 있습니다.

5. **자원순환 체계 구축**: 자원을 효율적으로 순환하고 관리하기 위해 분산형 폐기물 처리 시설을 구축하고, 시민의 참여를 통한 기후 행동을 활성화하는 포용적 거버넌스를 지향합니다.

6. **친환경 생태도시 조성**: 도심 내 생물 다양성을 확보하고, 지속 가능한 물순환 체계를 구축하여 사람과 자연이 공존하는 도시 환경을 만들기 위해 노력하고 있습니다.

이러한 다양한 정책을 통해 서울시는 온실가스를 적극적으로 저감하고, 기후변화에 대응할 수 있는 환경을 조성하려 하고 있습니다.


### 대화 기록 출력하기

In [17]:
for m in chat_history.messages:
    print(m)

content='서울시의 온실가스 저감 정책에 대해 알려줘.' additional_kwargs={} response_metadata={}
content='서울시는 온실가스 저감을 위해 여러 가지 정책과 전략을 추진하고 있습니다. 주요 내용은 다음과 같습니다:\n\n1. **탄소중립 목표 설정**: 2050년까지 탄소중립을 달성하기 위해 도시 인프라의 전환이 필요하며, 이를 위한 다양한 친환경 기술 개발 및 적용이 강조되고 있습니다.\n\n2. **건물 부문 탄소배출 감축**: 친환경 건축 기술을 개발하고 이를 적극적으로 적용하여 건물에서 발생하는 탄소배출을 줄이는 노력이 이루어지고 있습니다.\n\n3. **교통 및 에너지 혁신**: 미래 모빌리티 기술을 활용하고 친환경 차량 및 관련 인프라를 확충하여 교통 부문에서도 온실가스 배출을 감축할 계획입니다. 또한, 청정에너지를 기반으로 에너지 전환을 추진하고 있습니다.\n\n4. **대기 환경 관리**: 대기 환경을 고려한 공간계획과 배출원 관리체계를 강화하여 전체적인 대기 질 개선에 기여하고 있습니다.\n\n5. **자원순환 체계 구축**: 자원을 효율적으로 순환하고 관리하기 위해 분산형 폐기물 처리 시설을 구축하고, 시민의 참여를 통한 기후 행동을 활성화하는 포용적 거버넌스를 지향합니다.\n\n6. **친환경 생태도시 조성**: 도심 내 생물 다양성을 확보하고, 지속 가능한 물순환 체계를 구축하여 사람과 자연이 공존하는 도시 환경을 만들기 위해 노력하고 있습니다.\n\n이러한 다양한 정책을 통해 서울시는 온실가스를 적극적으로 저감하고, 기후변화에 대응할 수 있는 환경을 조성하려 하고 있습니다.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


## 질의 확장 구현하기

### 문자열 출력 파서 불러오기

In [18]:
from langchain_core.output_parsers import StrOutputParser

### 구체적인 질문으로 프롬프트 증강

In [19]:
query_for_nyc = "뉴욕은?"

# query augmentation 
# 기존 대화 내용을 활용해 query_augmentation 수행
query_augmentation_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="messages"), # 기존 대화 내용
        (
            "system",
            "기존의 대화 내용을 활용하여 사용자의 아래 질문의 의도를 파악하여 명료한 한 문장의 질문으로 변환하라. 대명사나 이, 저, 그와 같은 표현을 명확한 명사로 표현하라. :\n\n{query}",
        ),
    ]
)

### 결과 문자열로 반환하는 체인 생성

In [20]:
query_augmentation_chain = query_augmentation_prompt | chat | StrOutputParser()

In [21]:
augmented_query = query_augmentation_chain.invoke({
    "messages": chat_history.messages,
    "query": query_for_nyc  
})

print(augmented_query)

뉴욕시의 온실가스 저감 정책에 대해 알려줄 수 있나요?


### 벡터DB 검색 후 결과 출력

In [22]:
docs = retriever.invoke(augmented_query)

for d in docs:
    print(d)
    print('------')

page_content='New York City, because of its density and public transportation 
system, has long had a smaller per capita carbon footprint than 
any other big city in the United States — and we have made 
significant progress reducing GHG emissions over the last decade, 
using new technologies and innovations to get us there. The City 
has assumed a leading global role in fighting climate change, 
and the actions we take can become a national and global model. 
However, the lack of commitment by the federal government 
to the Paris Agreement has placed New York and the world in a 
precarious position. Time is running out.
While New York City has made strides to achieve a reduction in 
greenhouse gas emissions, global emissions continue to rise, 
putting New Yorkers at risk. 
CHANGE IN GHG EMISSIONS, 2005-2017
Source: Mayor’s Office, International Energy Agency
0%
5%
GLOBAL
NEW YORK CITY
-5%
15%
-15%
-20%
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
10%
-10%
20%
25%' 

### 언어 모델에서 답변 생성하기

In [23]:
chat_history.add_user_message(query_for_nyc) # query_for_nyc에 "뉴욕은?" 추가

answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer) 

print(answer)

뉴욕시는 온실가스 저감을 위해 다양한 정책과 프로그램을 시행하고 있습니다. 주요 내용은 다음과 같습니다:

1. **온실가스 감축 목표**: 뉴욕시는 2005년 대비 2030년까지 40% 감축, 2050년까지 탄소중립을 달성하는 것을 목표로 하고 있습니다.

2. **에너지 효율성 프로그램**: 뉴욕시는 에너지 효율적인 조치를 통해 2017년까지 시 전체 온실가스 배출량을 2005년 기준으로 17% 줄였으며, 시정부 건물의 온실가스 배출량은 거의 30% 감소했습니다. 또한, 시정부는 수십억 달러를 에너지 효율성 개선에 투자했습니다.

3. **새로운 건물에 대한 규정**: 2017년 로컬 법 32에 따라 새로운 건물들은 최신 에너지 효율 기준에 맞춰 건설되어야 하며, 2030년까지 모든 신축 건물은 넷 제로 에너지를 달성해야 합니다.

4. **청정 에너지와 기술 혁신**: 청정 에너지원으로의 전환과 새로운 기술의 도입을 통해 연료 사용을 줄이고, 온실가스 배출을 감소시키기 위해 노력하고 있습니다.

5. **커뮤니티 참여 및 교육**: GreeNYC 프로그램을 통해 시민들이 일상생활에서 탄소 발자국을 줄일 수 있는 방법을 배울 수 있도록 지원하며, 환경 정의 지침을 모든 프로젝트에 적용할 것을 권장합니다.

6. **교통 개선**: 대중교통 시스템의 개선과 친환경 모빌리티 솔루션을 통해 도시의 교통 부문에서 발생하는 온실가스 배출을 줄이는 방안을 추진하고 있습니다.

이런 다양한 정책과 프로그램을 통해 뉴욕시는 기후 변화에 대응하고 지속 가능한 도시 환경을 조성하려고 하고 있습니다.
